In [ ]:
import rasterio
import numpy as np
from PIL import Image
import os

# Input and output paths
TIF_PATH = r"C:\Users\SHEHER SAAZ\Music\Riphah University\Second Semester\Deep Learning\Assignment\Eurosatproject\data\Lahore_Dataset\lahore_rgb_2023.tif"
OUTPUT_DIR = r"C:\Users\SHEHER SAAZ\Music\Riphah University\Second Semester\Deep Learning\Assignment\Eurosatproject\data\Lahore_Dataset\patches"


os.makedirs(OUTPUT_DIR, exist_ok=True)

PATCH_SIZE = 64
patch_count = 0
skipped = 0

with rasterio.open(TIF_PATH) as src:
    print(f"Image size : {src.width} x {src.height} pixels")
    print(f"Bands      : {src.count}")
    print(f"CRS        : {src.crs}")

    # Read RGB bands (B4=band1, B3=band2, B2=band3 in your exported tif)
    img = src.read([1, 2, 3]).astype(np.float32)

    # Normalize to 0-255
    img = np.clip(img / 3000.0 * 255, 0, 255).astype(np.uint8)

    # Convert from (C, H, W) to (H, W, C)
    img = np.transpose(img, (1, 2, 0))

    h, w, _ = img.shape
    print(f"Total possible patches: {(h // PATCH_SIZE) * (w // PATCH_SIZE)}\n")

    for i in range(0, h - PATCH_SIZE + 1, PATCH_SIZE):
        for j in range(0, w - PATCH_SIZE + 1, PATCH_SIZE):
            patch = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

            # Skip patches that are mostly black (NoData areas)
            if np.mean(patch) < 5:
                skipped += 1
                continue

            patch_img = Image.fromarray(patch)
            patch_img.save(os.path.join(OUTPUT_DIR, f"lahore_patch_{patch_count:05d}.jpg"))
            patch_count += 1

print(f"✅ Saved patches : {patch_count}")
print(f"⏭️  Skipped (NoData): {skipped}")
print(f"📁 Output folder : {OUTPUT_DIR}")

RasterioIOError: Eurosatproject\data\Lahore_Dataset\lahore_rgb_2023.tif: No such file or directory